# Dataquest 2023 Final: fine-tuning TrOCR (license-plate checkpoint) on drug-label images (Kaggle)

In [ ]:
import numpy as np
import pandas as pd

DATA_DIR = "../data/ocr_prep"
MODEL_DIR = "../models/modelku"
SAMPLE_IMAGE_PATH = "../data/obat.jpg"

In [8]:
df = pd.read_csv(f"{DATA_DIR}/train.csv")
df['path'] = df['path'].str.replace('.png', '.jpg').str.replace('.jpeg', '.jpg')
df

,path,label
0,1.jpg,HOT IN CREAM
1,2.jpg,BYEBYE-FEVER
2,3.jpg,FITBES FORCE
3,4.jpg,TEMPRA PARACETAMOL
4,5.jpg,BIOBAIA INTERLAC
...,...,...
95,96.jpg,AMBEVEN
96,97.jpg,AMBROXOL HCL
97,98.jpg,AMCOR 10
98,99.jpg,AMITIZA LUBIPROSTONE


In [9]:
from transformers import TrOCRProcessor, VisionEncoderDecoderModel

# If you want to start from the pretrained model, load the checkpoint with `VisionEncoderDecoderModel`
processor = TrOCRProcessor.from_pretrained('microsoft/trocr-base-printed')

# TrOCR is a decoder model and should be used within a VisionEncoderDecoderModel
model = VisionEncoderDecoderModel.from_pretrained('DunnBC22/trocr-base-printed_license_plates_ocr')

Could not find image processor class in the image processor config or the model config. Loading based on pattern matching with the model's feature extractor configuration.


In [10]:
# set special tokens used for creating the decoder_input_ids from the labels
model.config.decoder_start_token_id = processor.tokenizer.cls_token_id
model.config.pad_token_id = processor.tokenizer.pad_token_id
# make sure vocab size is set correctly
model.config.vocab_size = model.config.decoder.vocab_size

# set beam search parameters
model.config.eos_token_id = processor.tokenizer.sep_token_id
model.config.max_length = 64
model.config.early_stopping = True
model.config.no_repeat_ngram_size = 3
model.config.length_penalty = 2.0
model.config.num_beams = 4

In [20]:
import torch
from torch.utils.data import Dataset
from PIL import Image

class IAMDataset(Dataset):
    def __init__(self, root_dir, df, processor, max_target_length=128):
        self.root_dir = root_dir
        self.df = df
        self.processor = processor
        self.max_target_length = max_target_length

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        # get file name + text 
        file_name = self.df['path'][idx]
        text = self.df['label'][idx]
        # prepare image (i.e. resize + normalize)
        image = Image.open(f"{DATA_DIR}/{file_name}").convert("RGB")
        pixel_values = self.processor(image, return_tensors="pt").pixel_values
        # add labels (input_ids) by encoding the text
        labels = self.processor.tokenizer(text, 
                                          padding="max_length", 
                                          max_length=self.max_target_length).input_ids
        # important: make sure that PAD tokens are ignored by the loss function
        labels = [label if label != self.processor.tokenizer.pad_token_id else -100 for label in labels]

        encoding = {"pixel_values": pixel_values.squeeze(), "labels": torch.tensor(labels)}
        return encoding

In [12]:
from sklearn.model_selection import train_test_split

# train_df, valid_df = train_test_split(df_all, test_size=0.04, random_state=0)
valid_df = df[80:]
train_df = df[:80]

train_df.reset_index(drop=True, inplace=True)
valid_df.reset_index(drop=True, inplace=True)

In [21]:
root_dir = ''
train_dataset = IAMDataset(root_dir, df=train_df, processor=processor)
eval_dataset = IAMDataset(root_dir, df=valid_df, processor=processor)

In [25]:
from datasets import load_metric
!pip install -q datasets jiwer
cer_metric = load_metric("cer")
def compute_metrics(pred):
    labels_ids = pred.label_ids
    pred_ids = pred.predictions

    pred_str = processor.batch_decode(pred_ids, skip_special_tokens=True)
    labels_ids[labels_ids == -100] = processor.tokenizer.pad_token_id
    label_str = processor.batch_decode(labels_ids, skip_special_tokens=True)

    cer = cer_metric.compute(predictions=pred_str, references=label_str)

    return {"cer": cer}

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [26]:
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments, default_data_collator
output_dir=DATA_DIR
training_args = Seq2SeqTrainingArguments(
    predict_with_generate=True,
    # evaluation_strategy="steps",
    evaluation_strategy="epoch",
    logging_strategy="epoch",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
#     fp16=True,
    overwrite_output_dir=True,
    output_dir=output_dir,
    # logging_steps=2,
    # save_steps=500,
    # eval_steps=200,
    report_to="none",
    num_train_epochs=5,
    # device="cuda",
)

# instantiate trainer
trainer = Seq2SeqTrainer(
    model=model,
    tokenizer=processor.feature_extractor,
    args=training_args,
    compute_metrics=compute_metrics,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=default_data_collator,
)

In [27]:
import time
train_time = time.time() ## in UTC time
trainer.train()
train_time = time.time() - train_time

print('Train time (sec)', train_time)

Epoch,Training Loss,Validation Loss,Cer
1,4.426400,4.759212,0.789030
2,3.451600,4.774612,0.789030
3,2.568500,5.270176,0.759494
4,1.904700,5.484310,0.755274
5,1.466500,5.507721,0.776371


Train time (sec) 2966.2740218639374


In [29]:
save_time = time.time()
model.save_pretrained(MODEL_DIR)
save_time = time.time() - save_time
print('save_time time (sec)', save_time)

save_time time (sec) 3.195984363555908


In [32]:
#@title Preparation Function
from transformers import TrOCRProcessor, VisionEncoderDecoderModel
from transformers import GenerationConfig
import os

model = VisionEncoderDecoderModel.from_pretrained('/kaggle/working/modelku').to("cpu")
generation_config = GenerationConfig(
    bos_token_id=0,
    decoder_start_token_id=0,
    early_stopping=True,
    eos_token_id=2,
    length_penalty=1.0,
    max_length=64,
    no_repeat_ngram_size=3,
    num_beams=4,
    pad_token_id=1,
    transformers_version="4.30.2",
    use_cache=False,
    return_dict_in_generate=True,
)

def inference(image):
  pixel_values = processor(image, return_tensors="pt").pixel_values.to("cpu")
  generated_ids = model.generate(pixel_values, generation_config=generation_config, output_scores=True)
  generated_text = processor.batch_decode(generated_ids[0], skip_special_tokens=True)[0]
  return generated_text

In [39]:
import cv2
image_path = SAMPLE_IMAGE_PATH
image = cv2.imread(image_path)
generated_text = inference(image)

# Print the generated text
print("Generated Text:", generated_text)

Generated Text: AEROL
